# 05_preprocessing_segmentation

Publication-ready Colab notebook for the equine thermography Explainable AI study. This notebook is designed to run after notebooks 00-04 and uses their outputs without changing the predefined train/valid/test split.

## Purpose
Prepare standardized image inputs for downstream classical and deep-learning analyses. The notebook uses folder-derived clinical labels from notebooks 01/04 and preserves expert hotspot flags from notebook 02. It does not use expert hotspot annotations as classification labels.

In [1]:
                    from pathlib import Path
import os, json, shutil, zipfile, hashlib, warnings, math, random
from datetime import datetime, timezone
import pandas as pd
import numpy as np

BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"
FEATURES_DIR = OUTPUT_ROOT / "features"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR,
          MODEL_SELECTION_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR, FEATURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Config dir:", CONFIG_DIR)

Project root: /content/project_thermography_equine
Config dir: /content/project_thermography_equine/outputs/config


In [4]:
required = [
    CONFIG_DIR / "analysis_config.json",
    CONFIG_DIR / "study_protocol.json",
    CONFIG_DIR / "master_metadata_qc.csv",
    CONFIG_DIR / "split_manifest.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 00-04 first. Missing:\n" + "\n".join(missing))

with open(CONFIG_DIR / "analysis_config.json", "r", encoding="utf-8") as f:
    analysis_config = json.load(f)
with open(CONFIG_DIR / "study_protocol.json", "r", encoding="utf-8") as f:
    study_protocol = json.load(f)

master = pd.read_csv(CONFIG_DIR / "master_metadata_qc.csv")
split_manifest = pd.read_csv(CONFIG_DIR / "split_manifest.csv")

required_cols = [
    "horse_id", "image_name", "split", "label_clinical", "label_binary",
    "relative_image_path", "included_in_final_analysis",
    "healthy_with_expert_hotspot", "annotation_label_conflict", "annotation_clinical_note"
]
missing_cols = [c for c in required_cols if c not in master.columns]
if missing_cols:
    raise KeyError("master_metadata_qc.csv is missing required columns from notebooks 02-04: " + ", ".join(missing_cols))

if master["annotation_label_conflict"].astype(bool).any():
    raise ValueError("True annotation-label conflicts are present. Resolve them before modeling.")

master = master[master["included_in_final_analysis"].astype(bool)].copy()
print("Included records:", len(master))
print("Healthy images with expert-marked hotspot retained:", int(master["healthy_with_expert_hotspot"].sum()))
display(master.groupby(["split", "label_clinical", "healthy_with_expert_hotspot"]).size().reset_index(name="n"))

Included records: 347
Healthy images with expert-marked hotspot retained: 6


,split,label_clinical,healthy_with_expert_hotspot,n
0,test,healthy,False,34
1,test,healthy,True,6
2,test,pathological,False,13
3,train,healthy,False,179
4,train,pathological,False,63
5,valid,healthy,False,38
6,valid,pathological,False,14


In [5]:

zip_candidates = sorted(
    list(BASE_DIR.glob("dataset_split*.zip")) +
    list(PROJECT_ROOT.glob("dataset_split*.zip")) +
    list(RAW_DATA_DIR.glob("dataset_split*.zip")),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
existing_images = []
for ext in image_exts:
    existing_images.extend(SPLIT_DATA_DIR.rglob(f"*{ext}"))
    existing_images.extend(SPLIT_DATA_DIR.rglob(f"*{ext.upper()}"))

if len(existing_images) == 0 and zip_candidates:
    print("Extracting dataset ZIP:", zip_candidates[0])
    with zipfile.ZipFile(zip_candidates[0], "r") as zf:
        zf.extractall(DATA_ROOT)


nested_candidates = [
    DATA_ROOT / "dataset_split" / "dataset_split",
    DATA_ROOT / "dataset_split-20260605T090743Z-3-001" / "dataset_split",
]
for nested in nested_candidates:
    if nested.exists() and nested.is_dir():
        for item in nested.iterdir():
            target = SPLIT_DATA_DIR / item.name
            if not target.exists():
                shutil.move(str(item), str(target))

existing_images = []
for ext in image_exts:
    existing_images.extend(SPLIT_DATA_DIR.rglob(f"*{ext}"))
    existing_images.extend(SPLIT_DATA_DIR.rglob(f"*{ext.upper()}"))
print("Images available in dataset_split:", len(existing_images))

def resolve_image_path(row):
    candidates = []
    for col in ["resolved_image_path", "image_path"]:
        if col in row and pd.notna(row[col]):
            candidates.append(Path(str(row[col])))
    rel = Path(str(row["relative_image_path"]))
    candidates.extend([
        SPLIT_DATA_DIR / rel,
        DATA_ROOT / "dataset_split" / rel,
        PROJECT_ROOT / rel,
        DATA_ROOT / rel,
    ])
    for p in candidates:
        if p.exists():
            return p
    return SPLIT_DATA_DIR / rel

master["resolved_image_path"] = master.apply(resolve_image_path, axis=1).astype(str)
missing_images = master[~master["resolved_image_path"].apply(lambda x: Path(x).exists())].copy()
print("Missing images:", len(missing_images))
if len(missing_images):
    display(missing_images[["split", "label_clinical", "image_name", "relative_image_path", "resolved_image_path"]].head(20))
    raise FileNotFoundError("Some images are missing. Upload/extract dataset_split before continuing.")

Extracting dataset ZIP: /content/dataset_split-20260605T090743Z-3-001.zip
Images available in dataset_split: 347
Missing images: 0


In [6]:
from PIL import Image, ImageOps

PREPROCESS_SIZE = int(os.environ.get("THERMO_PREPROCESS_SIZE", 224))
processed_rows = []

for _, row in master.iterrows():
    src = Path(row["resolved_image_path"])
    split = row["split"]
    label = row["label_clinical"]
    out_dir = CLEAN_IMAGE_DIR / split / label
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / row["image_name"]
    try:
        img = Image.open(src).convert("RGB")
        orig_w, orig_h = img.size

        img_resized = img.resize((PREPROCESS_SIZE, PREPROCESS_SIZE), resample=Image.BILINEAR)
        img_resized.save(out_path, quality=95)
        status = "ok"
        err = ""
    except Exception as e:
        orig_w = orig_h = np.nan
        status = "error"
        err = repr(e)
    r = row.to_dict()
    r.update({
        "preprocess_status": status,
        "preprocess_error": err,
        "original_width_preprocess": orig_w,
        "original_height_preprocess": orig_h,
        "preprocess_size": PREPROCESS_SIZE,
        "clean_image_path": str(out_path),
        "clean_relative_image_path": str(out_path.relative_to(PROJECT_ROOT)) if out_path.exists() else "",
    })
    processed_rows.append(r)

preprocess_manifest = pd.DataFrame(processed_rows)
if (preprocess_manifest["preprocess_status"] != "ok").any():
    display(preprocess_manifest[preprocess_manifest["preprocess_status"] != "ok"][["split", "image_name", "preprocess_error"]].head())
    raise RuntimeError("Some images failed preprocessing.")

print("Preprocessed images:", len(preprocess_manifest))
display(preprocess_manifest.groupby(["split", "label_clinical"]).size().reset_index(name="n"))

Preprocessed images: 347


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


In [7]:

preprocess_manifest.to_csv(CONFIG_DIR / "preprocess_manifest.csv", index=False)
preprocess_manifest.to_csv(METADATA_DIR / "preprocess_manifest.csv", index=False)
preprocess_manifest.to_csv(REPORTS_DIR / "preprocess_manifest.csv", index=False)

preprocess_report = pd.DataFrame([{
    "n_images": len(preprocess_manifest),
    "n_failed": int((preprocess_manifest["preprocess_status"] != "ok").sum()),
    "preprocess_size": PREPROCESS_SIZE,
    "healthy_with_expert_hotspot_retained": int(preprocess_manifest["healthy_with_expert_hotspot"].sum()),
    "color_jitter_used": False,
    "clinical_label_source": "folder_structure",
    "annotation_role": "localization_only"
}])
preprocess_report.to_csv(CONFIG_DIR / "preprocessing_report.csv", index=False)
preprocess_report.to_csv(REPORTS_DIR / "preprocessing_report.csv", index=False)

methods_preprocessing_text = (
    f"Images passing automated QC (n={len(preprocess_manifest)}) were deterministically resized to "
    f"{PREPROCESS_SIZE} x {PREPROCESS_SIZE} pixels for downstream modeling. No color jitter or temperature-palette "
    "altering augmentation was applied because pseudocolor encodes thermal information. Clinical labels were taken "
    "from the predefined folder structure, while expert hotspot annotations were retained only for localization analyses."
)
(CONFIG_DIR / "methods_preprocessing_text.txt").write_text(methods_preprocessing_text, encoding="utf-8")
(REPORTS_DIR / "methods_preprocessing_text.txt").write_text(methods_preprocessing_text, encoding="utf-8")
print(methods_preprocessing_text)

Images passing automated QC (n=347) were deterministically resized to 224 x 224 pixels for downstream modeling. No color jitter or temperature-palette altering augmentation was applied because pseudocolor encodes thermal information. Clinical labels were taken from the predefined folder structure, while expert hotspot annotations were retained only for localization analyses.


In [8]:
from pathlib import Path
import shutil

clean_images_dir = DATA_ROOT / "processed" / "clean_images"
zip_base = DATA_ROOT / "processed" / "clean_images_224x224"

if not clean_images_dir.exists():
    raise FileNotFoundError(f"Missing clean images directory: {clean_images_dir}")

n_images = len(list(clean_images_dir.rglob("*.jpg"))) + len(list(clean_images_dir.rglob("*.png")))

print("Clean images directory:", clean_images_dir)
print("Images found:", n_images)

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=clean_images_dir.parent,
    base_dir=clean_images_dir.name
)

print("Saved ZIP:", zip_path)

Clean images directory: /content/project_thermography_equine/data/processed/clean_images
Images found: 347
Saved ZIP: /content/project_thermography_equine/data/processed/clean_images_224x224.zip
